In [ ]:
# @title License
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Aerial and Satellite Models: Region Search with BigQuery Vector Search

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fearth_ai_remote_sensing%2Fnotebooks%2Fregion_search_bigquery%2Fearth_ai_region_search_bigquery.ipynb?utm_source=earth_ai_remote_sensing_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/googlemaps-samples/insights-samples/blob/main/earth_ai_remote_sensing/notebooks/region_search_bigquery/earth_ai_region_search_bigquery.ipynb&utm_source=earth_ai_remote_sensing_notebooks">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/googlemaps-samples/insights-samples/main/earth_ai_remote_sensing/notebooks/region_search_bigquery/earth_ai_region_search_bigquery.ipynb&utm_source=earth_ai_remote_sensing_notebooks">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Workbench logo"><br> Run in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/googlemaps-samples/insights-samples/blob/main/earth_ai_remote_sensing/notebooks/region_search_bigquery/earth_ai_region_search_bigquery.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>
<br clear="all"/>

## Overview

### Overall Goal
This notebook provides a production-ready blueprint for **metro-scale overhead imagery discovery and opportunity analysis** using **Google Maps Platform Aerial and Satellite Models (`ASM - Classification and Retrieval`)** on **Agent Platform Model Garden** (part of the **Google Earth AI** effort) together with **BigQuery Vector Search**:
1. **Native-Resolution Tiling (`GeoTIFF` & Public-Domain `USGS NAIP`)** slices customer Cloud-Optimized GeoTIFFs (`gs://.../*.tif`) at **100% native sensor ground sample distance (GSD)** (`224×224` or `512×512 px`, avoiding aspect/resize distortion) or tiles a bounding box from federal **USDA/USGS NAIP (`USGSNAIPImagery`)** (`0.6 m` GSD).
2. **Aerial and Satellite Models (`ASM - Classification and Retrieval`, `RS-SigLIP2`) on Agent Platform Model Garden** — deployed via [`google-research/remote-sensing`](https://github.com/google-research/remote-sensing/blob/main/remote_sensing/vertex_ai/utils.py) `remote_sensing.vertex_ai.utils` — encode overhead imagery tiles and natural-language prompts into a shared, L2-normalized **`1152-D` vector space**.
3. **BigQuery (`VECTOR_SEARCH` & `ML.DISTANCE`)** indexes tens of thousands to millions of regional orthoimagery tiles and executes sub-second **Text-to-Tile Semantic Search**, **Tile-to-Tile Visual Similarity**, and **Opportunity Finder (Vector Algebra / Negative Background Subtraction)** queries in standard SQL (`$0/hr` idle GPU cost).
4. **Interactive Google Maps Visualization (`google.maps.Map`)** overlays the discovered tile footprints onto an interactive Google Maps Satellite/Hybrid view for visual inspection.

---

### Architecture & Division of Labor

| Stage | Component | What It Does | Frequency & Cost Profile |
| :--- | :--- | :--- | :--- |
| **1. Native-GSD Tile Grid** | **`rasterio` Windowed Tiler / USDA NAIP (`USGSNAIPImagery`)** | Slices GeoTIFFs at native sensor resolution (`0.1–0.3 m/px` sweet spot; supports `0.3–3 m/px`) or `0.6 m` NAIP orthoimagery into `224×224` or `512×512 px` chips and `.jsonl` shards. | **One-time / Periodic** refresh. |
| **2. Feature Extraction** | **ASM - Classification and Retrieval (`RS-SigLIP2`)** *(Agent Platform Model Garden)* | Uses `remote_sensing.vertex_ai.utils` (`create_model` / `deploy_model` / `batch_predict`) to encode overhead tiles (`IMAGE_ONLY`) and text prompts (`TEXT_ONLY`) into L2-normalized `ARRAY<FLOAT64>` (`1152-D`) vectors. | **One-time GPU job** for tiles; **< 50 ms** for new text prompts. |
| **3. Spatial & Vector Indexing** | **BigQuery (`GEOGRAPHY` + `CLUSTER BY` + `VECTOR INDEX`)** | Stores tile bounding boxes, `GEOGRAPHY` polygons, `1152-D` image vectors, and pre-encoded text vectors in native BigQuery tables. | **Serverless Storage** (`$0.02 / GB / month`). |
| **4. Metro-Scale Retrieval** | **BigQuery `ML.DISTANCE` & `VECTOR_SEARCH`** | Computes cosine similarity (`1 - ML.DISTANCE(img, txt, 'COSINE')`), regional z-scores (`+σ`), and composite vector algebra (`pos - α·neg`). | **Pure SQL** (`$0/hr` idle GPU cost once indexed). |

---

### Terms of Use & Public-Domain Data Provenance
- **Overhead Imagery for Model Inference (`17 U.S.C. § 105`)**: All sample overhead imagery tiles passed into `ASM - Classification and Retrieval` (`RS-SigLIP2`) in this notebook are sourced from the **U.S. Department of Agriculture (USDA) / USGS National Agriculture Imagery Program (`USGSNAIPImagery`)** (`0.6 m` GSD, `512×512 px`), which is a work of the U.S. Federal Government in the **public domain** (or from your own licensed GeoTIFFs in Cloud Storage).
- **Google Maps Platform Terms of Service (Section 3.2.3)**: The **Google Maps JavaScript API** is used strictly as an **interactive visual basemap** in Step 6 and is **never** scraped, cached, or passed as input into any ML model.

---

### How to Use This Notebook
1. **Run Step 1 (`Setup & Authentication`)** to install dependencies (`google-cloud-aiplatform`, `requests-toolbelt`, `rasterio`, `google-cloud-bigquery`), authenticate with Google Cloud, and clone [`google-research/remote-sensing`](https://github.com/google-research/remote-sensing).
2. **Configure Step 2 (`Configuration & All Notebook Parameters`)** — **all user-configurable `# @param` parameters for the entire notebook are consolidated in that single cell** at the top of the notebook.
3. **Run Steps 3–7 sequentially** to deploy or attach to an **ASM - Classification and Retrieval (`RS-SigLIP2`)** endpoint (or query pre-indexed BigQuery tables directly), tile GeoTIFFs or NAIP AOIs into BigQuery (**Step 3B**), and run **Text-to-Tile Search**, **Tile-to-Tile Visual Similarity**, **Opportunity Finder Vector Algebra**, and **Interactive Google Maps Visualization**.

## Step 1: Setup & Authentication

First, install the required Google Cloud, Agent Platform, BigQuery, and plotting libraries, clone the official [`google-research/remote-sensing`](https://github.com/google-research/remote-sensing) repository (which provides [`remote_sensing.vertex_ai.utils`](https://github.com/google-research/remote-sensing/blob/main/remote_sensing/vertex_ai/utils.py)), and authenticate your environment:

- **Colab Enterprise / Workbench / BigQuery Studio**: Uses your attached runtime credentials automatically via `google.auth.default()`.
- **Public Google Colab**: Prompts for browser authentication via `google.colab.auth.authenticate_user()` and automatically reads `GCP_PROJECT_ID` and `GMP_API_KEY` from `google.colab.userdata` secrets if configured.

In [ ]:
# @title 1. Install Dependencies, Clone `google-research/remote-sensing` & Authenticate
!pip install --quiet --upgrade google-cloud-aiplatform requests-toolbelt google-cloud-bigquery google-cloud-storage db-dtypes rasterio pandas matplotlib pillow requests

import base64
import concurrent.futures as cf
from datetime import date
import io
import json
import math
import os
import subprocess
import sys
import uuid
import warnings
from typing import Any, Dict, List, Optional, Tuple

from google.cloud import aiplatform
from google.cloud import bigquery
from google.cloud import storage
from IPython.display import HTML, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageStat
import rasterio
from rasterio.warp import transform_bounds
from rasterio.windows import Window
import requests

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 1000)

# Clone official Earth AI Remote Sensing repository for remote_sensing.vertex_ai.utils
if not os.path.exists("remote-sensing"):
    subprocess.run(
        ["git", "clone", "--quiet", "https://github.com/google-research/remote-sensing.git"],
        check=True,
    )
repo_path = os.path.abspath("remote-sensing")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

from remote_sensing.vertex_ai import utils

# Dual-mode authentication (Colab userdata / interactive auth + Enterprise ADC fallback)
_COLAB_SECRET_PROJECT = ""
_COLAB_SECRET_GMP_KEY = ""
try:
    from google.colab import auth as colab_auth
    from google.colab import userdata

    colab_auth.authenticate_user()
    try:
        _COLAB_SECRET_PROJECT = userdata.get("GCP_PROJECT_ID") or ""
    except Exception:
        pass
    try:
        _COLAB_SECRET_GMP_KEY = userdata.get("GMP_API_KEY") or ""
    except Exception:
        pass
    print("✅ Authenticated via Google Colab user session.")
except ImportError:
    import google.auth

    _, _adc_project = google.auth.default()
    _COLAB_SECRET_PROJECT = _adc_project or ""
    print("✅ Authenticated via Application Default Credentials (Colab Enterprise / Workbench).")

## Step 2: Configuration & All Notebook Parameters

All configurable parameters for the entire notebook are consolidated in the single form below so you can customize your run in one place before executing the workflow:

| Parameter Group | Key Parameters | Purpose |
| :--- | :--- | :--- |
| **1. Google Cloud, BigQuery & Google Maps** | `PROJECT_ID`, `REGION`, `BQ_DATASET`, `TILES_TABLE_NAME`, `QUERIES_TABLE_NAME`, `GMP_API_KEY` | Selects your GCP project, BigQuery dataset/tables (`region_tiles_la` or `region_tiles_sf`), and Google Maps JavaScript API key (`GMP_API_KEY`) for Step 6. |
| **2. AOI & Public-Domain NAIP Grid** | `AOI_NAME`, `AOI_MIN_LAT`, `AOI_MIN_LNG`, `AOI_MAX_LAT`, `AOI_MAX_LNG`, `MAX_DEMO_TILES`, `FETCH_LIVE_NAIP_TILES` | Defines the bounding box for live USGS NAIP (`0.6 m` GSD, `512×512 px`) tile fetching when building a new table from scratch. |
| **3. Agent Platform Model Garden (`remote_sensing.vertex_ai.utils`)** | `DEPLOY_LIVE_ENDPOINT`, `EXISTING_ENDPOINT_ID`, `MODEL_TYPE`, `MODEL_MODE`, `ACCELERATOR`, `USE_DEDICATED_ENDPOINT`, `MIN_REPLICA_COUNT`, `MAX_REPLICA_COUNT`, `RUN_BATCH_JOB_EXAMPLE` | Configures `utils.create_model(...)` and `utils.deploy_model(...)` from [`remote_sensing/vertex_ai/utils.py`](https://github.com/google-research/remote-sensing/blob/main/remote_sensing/vertex_ai/utils.py) for the **RS-SigLIP2** (`SIGLIP2_512_RC1` / `SIGLIP2_224_RC1`) Aerial and Satellite Model, or runs immediately against pre-computed BigQuery tables (`DEPLOY_LIVE_ENDPOINT = False`). |
| **4. Search & Vector Algebra** | `SEARCH_QUERY_ID`, `SEED_TILE_ID`, `POSITIVE_QUERY_ID`, `NEGATIVE_QUERY_ID`, `NEGATIVE_WEIGHT`, `TOP_K_RESULTS` | Configures the prompts, seed tile, and composite vector algebra weights for **Step 5 (5A, 5B, and 5C)**. |
| **5. Resource Cleanup** | `UNDEPLOY_ENDPOINT_ON_FINISH`, `DELETE_ENDPOINT_AND_MODEL` | Controls automatic GPU endpoint teardown in **Step 7**. |

In [ ]:
# @title 2. Configuration & All Notebook Parameters

# @markdown ### 1. Google Cloud, BigQuery & Google Maps Settings
PROJECT_ID = ""  # @param {type:"string"}
REGION = "us-central1"  # @param ["us-central1", "us-east4", "us-west1", "europe-west4"]
BQ_DATASET = "rsfm_demo"  # @param {type:"string"}
TILES_TABLE_NAME = "region_tiles_la"  # @param ["region_tiles_la", "region_tiles_sf"] {allow-input: true}
QUERIES_TABLE_NAME = "region_queries_la"  # @param ["region_queries_la", "region_queries_sf"] {allow-input: true}
GMP_API_KEY = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 2. Native-Resolution Tiling Settings (Bring-Your-Own GeoTIFF or Public-Domain USGS NAIP Grid)
# @markdown Set `BUILD_TILES_FROM_SCRATCH = True` to run the end-to-end Step 3B tiler: slices `SOURCE_GEOTIFF_URI` (`gs://.../*.tif`) at 100% native sensor resolution (`224x224` or `512x512` windows), or slices the AOI bounding box below from federal `USGSNAIPImagery` (`0.6m` GSD), generates `.jsonl` batch shards, and populates BigQuery when an endpoint is attached.
BUILD_TILES_FROM_SCRATCH = False  # @param {type:"boolean"}
SOURCE_GEOTIFF_URI = ""  # @param {type:"string"}
CAPTURE_DATE_ISO = "2022-05-11"  # @param {type:"string"}
AOI_NAME = "Port of Los Angeles / San Pedro"  # @param {type:"string"}
AOI_MIN_LAT = 33.7250  # @param {type:"number"}
AOI_MIN_LNG = -118.2800  # @param {type:"number"}
AOI_MAX_LAT = 33.7400  # @param {type:"number"}
AOI_MAX_LNG = -118.2600  # @param {type:"number"}
TILE_SIZE_PX = 512  # @param [224, 512] {type:"raw"}
TARGET_GSD_M = 0.6  # @param {type:"number"}
MAX_DEMO_TILES = 32  # @param {type:"slider", min:4, max:256, step:4}
MIN_TILE_STDDEV = 6.0  # @param {type:"number"}
FETCH_LIVE_NAIP_TILES = False  # @param {type:"boolean"}
CREATE_VECTOR_INDEX = False  # @param {type:"boolean"}

# @markdown ---
# @markdown ### 3. Aerial and Satellite Models (`ASM - Classification and Retrieval`) — Agent Platform Model Garden Settings
# @markdown Set `DEPLOY_LIVE_ENDPOINT = False` (default) to immediately run BigQuery `VECTOR_SEARCH` / `ML.DISTANCE` over pre-computed `1152-D` `RS-SigLIP2` embeddings without waiting ~10 minutes for a GPU endpoint deployment. Set to `True` to deploy a live endpoint via `utils.create_model` & `utils.deploy_model`, or attach to `EXISTING_ENDPOINT_ID`.
DEPLOY_LIVE_ENDPOINT = False  # @param {type:"boolean"}
EXISTING_ENDPOINT_ID = ""  # @param {type:"string"}
DISPLAY_NAME = "asm-classification-retrieval"  # @param {type:"string"}
MODEL_TYPE = "SIGLIP2_512_RC1"  # @param ["SIGLIP2_224_RC1", "SIGLIP2_512_RC1"]
MODEL_MODE = "COMBINED"  # @param ["IMAGE_ONLY", "TEXT_ONLY", "COMBINED"]
ACCELERATOR = "NVIDIA_L4"  # @param ["CPU", "NVIDIA_L4", "NVIDIA_TESLA_A100", "NVIDIA_A100_80GB"]
USE_DEDICATED_ENDPOINT = True  # @param {type:"boolean"}
SERVICE_ACCOUNT = ""  # @param {type:"string"}
MIN_REPLICA_COUNT = 1  # @param {type:"integer"}
MAX_REPLICA_COUNT = 1  # @param {type:"integer"}
RUN_BATCH_JOB_EXAMPLE = False  # @param {type:"boolean"}
GCS_BUCKET_NAME = ""  # @param {type:"string"}
INPUT_SOURCE_FORMAT = "file-list"  # @param ["file-list", "jsonl"]
BATCH_SIZE = 8  # @param {type:"integer"}

# @markdown ---
# @markdown ### 4. Search, Visual Similarity & Opportunity Finder (Vector Algebra) Parameters
SEARCH_QUERY_ID = "distribution_warehouses"  # @param ["solar_rooftops", "distribution_warehouses", "shipping_containers", "Gantry_cranes", "tennis_courts", "swimming_pools", "parking_lots", "construction_sites", "water_treatment"]
SEED_TILE_ID = "r038_c052"  # @param {type:"string"}
POSITIVE_QUERY_ID = "distribution_warehouses"  # @param ["distribution_warehouses", "parking_lots", "shipping_containers", "construction_sites", "Gantry_cranes"]
NEGATIVE_QUERY_ID = "solar_rooftops"  # @param ["solar_rooftops", "swimming_pools", "tennis_courts", "water_treatment"]
NEGATIVE_WEIGHT = 0.5  # @param {type:"slider", min:0.0, max:1.0, step:0.1}
TOP_K_RESULTS = 10  # @param {type:"slider", min:5, max:25, step:5}
VISUAL_GRID_TILES = 6  # @param {type:"slider", min:3, max:9, step:3}

# @markdown ---
# @markdown ### 5. Resource Cleanup Settings
UNDEPLOY_ENDPOINT_ON_FINISH = False  # @param {type:"boolean"}
DELETE_ENDPOINT_AND_MODEL = False  # @param {type:"boolean"}

# Resolve fallbacks from Colab secrets or Application Default Credentials
if not PROJECT_ID:
    PROJECT_ID = _COLAB_SECRET_PROJECT or os.environ.get("GOOGLE_CLOUD_PROJECT", "")
if not GMP_API_KEY:
    GMP_API_KEY = _COLAB_SECRET_GMP_KEY or os.environ.get("GMP_API_KEY", "")

if not PROJECT_ID:
    raise ValueError(
        "Please set PROJECT_ID in Step 2 (Configuration) or configure GCP_PROJECT_ID in Colab Secrets."
    )

TILES_TABLE_FQN = f"{PROJECT_ID}.{BQ_DATASET}.{TILES_TABLE_NAME}"
QUERIES_TABLE_FQN = f"{PROJECT_ID}.{BQ_DATASET}.{QUERIES_TABLE_NAME}"

aiplatform.init(project=PROJECT_ID, location=REGION)
bq_client = bigquery.Client(project=PROJECT_ID)

selected_model_cfg = utils.MODEL_CONFIGS[MODEL_TYPE]
selected_platform_cfg = utils.PLATFORM_CONFIGS[ACCELERATOR]

config_summary_df = pd.DataFrame(
    [
        {"Setting": "Google Cloud Project & Region", "Value": f"{PROJECT_ID} ({REGION})"},
        {"Setting": "BigQuery Tiles Table", "Value": TILES_TABLE_FQN},
        {"Setting": "BigQuery Queries Table", "Value": QUERIES_TABLE_FQN},
        {
            "Setting": "Tiling Source & Native Window",
            "Value": (
                f"GeoTIFF `{SOURCE_GEOTIFF_URI}` ({TILE_SIZE_PX}x{TILE_SIZE_PX} native px)"
                if SOURCE_GEOTIFF_URI
                else f"USGS NAIP `{AOI_NAME}` ({TILE_SIZE_PX}x{TILE_SIZE_PX} px @ {TARGET_GSD_M}m GSD, BUILD={BUILD_TILES_FROM_SCRATCH})"
            ),
        },
        {
            "Setting": "ASM Model (`utils.MODEL_CONFIGS`)",
            "Value": f"{MODEL_TYPE} (ASM - Classification and Retrieval / RS-SigLIP2) -> id={selected_model_cfg.model_id}, mode={MODEL_MODE}",
        },
        {
            "Setting": "Platform Config (`utils.PLATFORM_CONFIGS`)",
            "Value": f"{ACCELERATOR} -> machine_type={selected_platform_cfg.machine_type}",
        },
        {
            "Setting": "Agent Platform Endpoint Mode",
            "Value": (
                f"Attach to {EXISTING_ENDPOINT_ID}"
                if EXISTING_ENDPOINT_ID
                else ("Deploy New Endpoint" if DEPLOY_LIVE_ENDPOINT else "Pre-computed BigQuery Mode (Fast)")
            ),
        },
        {
            "Setting": "Active Search & Algebra Config",
            "Value": f"Query='{SEARCH_QUERY_ID}' | Seed='{SEED_TILE_ID}' | Algebra='{POSITIVE_QUERY_ID} - {NEGATIVE_WEIGHT}*{NEGATIVE_QUERY_ID}'",
        },
    ]
)
print("✅ Configuration validated:")
display(config_summary_df)

## Step 3: Aerial and Satellite Models (`ASM - Classification and Retrieval`) on Agent Platform Model Garden

### About Aerial and Satellite Models & `RS-SigLIP2`
Google Maps Platform's **Aerial and Satellite Models (`ASM`)** are part of the **Google Earth AI** effort and are available in **Agent Platform Model Garden**. This notebook uses **`ASM - Classification and Retrieval`** (**`RS-SigLIP2`**, `SIGLIP2_512_RC1` / `SIGLIP2_224_RC1`), a two-tower vision-language foundation model trained on overhead aerial and satellite imagery using a Sigmoid Loss for Language-Image Pre-Training (SigLIP 2) objective.

### Using `remote_sensing.vertex_ai.utils` for Model Creation & Deployment
All model metadata, container URIs (`utils.SERVE_DOCKER_URI`), checkpoint paths (`utils.MODEL_CONFIGS`), and hardware profiles (`utils.PLATFORM_CONFIGS`) are managed by the official [`remote_sensing/vertex_ai/utils.py`](https://github.com/google-research/remote-sensing/blob/main/remote_sensing/vertex_ai/utils.py) module cloned in Step 1:

- **`utils.create_model(...)`**: Registers **`ASM - Classification and Retrieval`** (`SIGLIP2_512_RC1` or `SIGLIP2_224_RC1`) with the official serving container (`us-docker.pkg.dev/vertex-ai-restricted/vertex-vision-model-garden-remote-sensing-dockers/remote-sensing-serve-tf-gpu:latest`) and environment variables (`MODEL_ID`, `MODEL_PATH`, `MODEL_TYPE`, `MODEL_MODE`, `PLATFORM`). Passing `batch_model=True` prepares the model for `model.batch_predict(...)`.
- **`utils.deploy_model(...)`**: Creates an Online Prediction Endpoint (`dedicated_endpoint_enabled=USE_DEDICATED_ENDPOINT`) and deploys the model onto the hardware profile from `utils.PLATFORM_CONFIGS[ACCELERATOR]` (e.g., `g2-standard-8` with `1× NVIDIA_L4`).
- **`utils.b64_png(image)`**: Encodes `PIL.Image` patches to base64 PNG strings for `{"image": utils.b64_png(img)}`, `{"text": prompt}`, or `{"image": utils.b64_png(img), "texts": [...]}` requests.

> **Best Practice — Why We Rank by Cosine Similarity (`1 - ML.DISTANCE`) Instead of Raw `similarity_scores`**:
> In `COMBINED` mode, `similarity_scores` applies an uncalibrated sigmoid (`1 / (1 + exp(-dot))`). Storing the L2-normalized `1152-D` `image_embedding` and `text_embedding` vectors in BigQuery and ranking by **cosine similarity (`1 - ML.DISTANCE(img, txt, 'COSINE')`)** plus **regional z-score (`+σ` above the metro median)** provides well-calibrated separation (`+2.5σ to +6.0σ`) across tens of thousands of tiles.

In [ ]:
# @title 3. Deploy/Attach ASM - Classification and Retrieval (`RS-SigLIP2`) via `remote_sensing.vertex_ai.utils`
endpoint: Optional[aiplatform.Endpoint] = None
model: Optional[aiplatform.Model] = None
unique_suffix = uuid.uuid4().hex[:6]

if EXISTING_ENDPOINT_ID:
    endpoint = aiplatform.Endpoint(EXISTING_ENDPOINT_ID)
    print(f"✅ Attached to existing Endpoint: {endpoint.resource_name}")
elif DEPLOY_LIVE_ENDPOINT:
    print(
        f"🚀 Creating & deploying {MODEL_TYPE} ({MODEL_MODE}) on {ACCELERATOR} "
        "via remote_sensing.vertex_ai.utils..."
    )
    model = utils.create_model(
        display_name=f"{DISPLAY_NAME}-{MODEL_TYPE.lower()}-{MODEL_MODE.lower()}-{ACCELERATOR.lower()}-{unique_suffix}",
        model_type=MODEL_TYPE,
        model_mode=MODEL_MODE,
        accelerator=ACCELERATOR,
        description=f"Model {DISPLAY_NAME} ({MODEL_TYPE}/{MODEL_MODE} on {ACCELERATOR})",
    )
    print(f"Created model: {model.resource_name} ({model.display_name})")

    endpoint = utils.deploy_model(
        endpoint_name=f"{DISPLAY_NAME}-{unique_suffix}-endpoint",
        model=model,
        accelerator=ACCELERATOR,
        service_account=SERVICE_ACCOUNT if SERVICE_ACCOUNT else None,
        min_replica_count=MIN_REPLICA_COUNT,
        max_replica_count=MAX_REPLICA_COUNT,
        use_dedicated_endpoint=USE_DEDICATED_ENDPOINT,
    )
    print(f"✅ Deployed live endpoint: {endpoint.resource_name}")
else:
    print(
        "ℹ️ DEPLOY_LIVE_ENDPOINT is False — skipping online GPU deployment and using "
        f"BigQuery tables `{TILES_TABLE_FQN}` and `{QUERIES_TABLE_FQN}` (or serverless Batch Prediction)."
    )

# Optional: Smoke-test live online inference using the official remote_sensing.vertex_ai.utils protocol
if endpoint is not None:
    sample_prompts = [
        "solar panels on warehouse roof",
        "shipping containers and intermodal freight yard",
    ]
    sample_img = Image.new("RGB", (TILE_SIZE_PX, TILE_SIZE_PX), color=(128, 136, 120))

    if MODEL_MODE in ("IMAGE_ONLY", "COMBINED"):
        img_result = endpoint.predict(
            instances=[{"image": utils.b64_png(sample_img)}],
            parameters={"batch_size": 1},
            use_dedicated_endpoint=USE_DEDICATED_ENDPOINT,
        )
        img_emb = img_result.predictions[0]["image_embedding"]
        print(f"  • Image embedding length: {len(img_emb)}, L2 norm: {np.linalg.norm(img_emb):.4f}")

    if MODEL_MODE in ("TEXT_ONLY", "COMBINED"):
        txt_result = endpoint.predict(
            instances=[{"text": prompt} for prompt in sample_prompts],
            parameters={"batch_size": len(sample_prompts)},
            use_dedicated_endpoint=USE_DEDICATED_ENDPOINT,
        )
        txt_emb = txt_result.predictions[0]["text_embedding"]
        print(f"  • Text embedding length: {len(txt_emb)}, L2 norm: {np.linalg.norm(txt_emb):.4f}")

## Step 3B: Native-Resolution Tiling & BigQuery Vector Indexing (`GeoTIFF` or `USGS NAIP`)

Before querying a new region in BigQuery, raw overhead imagery must be sliced into fixed-pixel chips, embedded into **`1152-D`** unit vectors, and loaded into BigQuery. Two engineering practices are critical for **`ASM - Classification and Retrieval` (`RS-SigLIP2`)**:

1. **Preserve Native Sensor Resolution (Zero Resize Distortion)**:
   - Near-nadir `0.1–0.3 m/px` aerial imagery is the primary sweet spot (`0.3–3 m/px` supported).
   - Always slice exact **`224×224`** (`SIGLIP2_224_RC1`) or **`512×512`** (`SIGLIP2_512_RC1`) pixel windows directly from the source Cloud-Optimized GeoTIFF (`rasterio.windows.Window(col_off, row_off, tile_px, tile_px)`) rather than feeding arbitrary rectangle dimensions that trigger server-side resize scaling.
2. **Two Production Ingestion Modes**:
   - **Million-Tile Scale (Serverless Batch Prediction)**: `stage_batch_prediction_inputs(...)` uploads PNG chips to Cloud Storage and writes a `file-list` manifest (`input_uris.txt`) for `batch_model.batch_predict(..., instances_format="file-list")`, and `ingest_batch_predictions_to_bigquery(...)` maps each output `instance` URI (`gs://.../chips/{tile_id}.png`) back to its spatial metadata in a partitioned & clustered BigQuery table (`PARTITION BY capture_date CLUSTER BY zone, tile_id`) + optional `CREATE VECTOR INDEX`.
   - **Interactive AOI Scale (Online Dedicated Endpoint)**: When `BUILD_TILES_FROM_SCRATCH = True` and an online `endpoint` is attached in Step 3, `embed_online_and_load_bigquery(...)` chunks tiles in power-of-two batches (`1, 2, 4, 8` chips, `< 1.5 MB` request cap) and populates `TILES_TABLE_FQN` and `QUERIES_TABLE_FQN` directly.

In [ ]:
# @title 3B. Tile GeoTIFF / NAIP AOI at Native GSD, Stage Batch Inputs & Populate BigQuery
import contextlib
import google.auth
import google.auth.transport.requests

USGS_NAIP_IMAGERY_EXPORT_URL = (
    "https://imagery.nationalmap.gov/arcgis/rest/services/USGSNAIPImagery/ImageServer/exportImage"
)
PUBLIC_NAIP_STAC_SEARCH_URL = "https://planetarycomputer.microsoft.com/api/stac/v1/search"
PUBLIC_NAIP_STAC_SIGN_URL = "https://planetarycomputer.microsoft.com/api/sas/v1/sign"
_WEB_MERCATOR_R = 6378137.0

DEFAULT_DOMAIN_QUERIES: List[Dict[str, str]] = [
    {
        "query_id": "distribution_warehouses",
        "label": "Distribution Warehouses",
        "prompt": "large commercial distribution warehouse buildings with loading docks",
    },
    {
        "query_id": "solar_rooftops",
        "label": "Rooftop Solar Arrays",
        "prompt": "rooftop solar photovoltaic panel arrays on commercial buildings",
    },
    {
        "query_id": "shipping_containers",
        "label": "Shipping Containers",
        "prompt": "stacked intermodal shipping containers in a port or freight yard",
    },
    {
        "query_id": "Gantry_cranes",
        "label": "Port Gantry Cranes",
        "prompt": "container port terminal with ship-to-shore gantry cranes",
    },
    {
        "query_id": "parking_lots",
        "label": "Surface Parking Lots",
        "prompt": "large paved surface parking lot with painted stalls and parked cars",
    },
    {
        "query_id": "construction_sites",
        "label": "Active Construction Sites",
        "prompt": "active construction site with graded earth and building foundations",
    },
    {
        "query_id": "tennis_courts",
        "label": "Tennis Courts",
        "prompt": "outdoor hardcourt tennis courts with painted boundary lines",
    },
    {
        "query_id": "swimming_pools",
        "label": "Swimming Pools",
        "prompt": "outdoor rectangular or residential swimming pools with blue water",
    },
    {
        "query_id": "water_treatment",
        "label": "Wastewater Clarifiers",
        "prompt": "wastewater treatment plant with circular clarifier tanks",
    },
]


def _rasterio_gcs_env(uri: str = "") -> rasterio.Env:
    """Authenticates GDAL `/vsigs/` reads using active Colab / ADC credentials (only for `gs://` URIs)."""
    extra: Dict[str, str] = {}
    if uri.startswith(("gs://", "/vsigs/")):
        creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/devstorage.read_only"])
        if not creds.valid:
            creds.refresh(google.auth.transport.requests.Request())
        if creds.token:
            extra["GDAL_HTTP_HEADERS"] = f"Authorization: Bearer {creds.token}"
    return rasterio.Env(
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        **extra,
    )


def _lng_to_merc_x(lng: float) -> float:
    return _WEB_MERCATOR_R * math.radians(lng)


def _lat_to_merc_y(lat: float) -> float:
    return _WEB_MERCATOR_R * math.log(math.tan(math.pi / 4.0 + math.radians(lat) / 2.0))


def _merc_x_to_lng(x: float) -> float:
    return math.degrees(x / _WEB_MERCATOR_R)


def _merc_y_to_lat(y: float) -> float:
    return math.degrees(2.0 * math.atan(math.exp(y / _WEB_MERCATOR_R)) - math.pi / 2.0)


def _bbox_wkt_polygon(min_lng: float, min_lat: float, max_lng: float, max_lat: float) -> str:
    return (
        f"POLYGON(({min_lng:.7f} {min_lat:.7f}, {max_lng:.7f} {min_lat:.7f}, "
        f"{max_lng:.7f} {max_lat:.7f}, {min_lng:.7f} {max_lat:.7f}, {min_lng:.7f} {min_lat:.7f}))"
    )


def discover_public_naip_cog_stac(
    min_lng: float, min_lat: float, max_lng: float, max_lat: float, datetime_range: str = "2020-01-01/2025-12-31"
) -> Tuple[str, str]:
    """Discovers & signs the latest public-domain USDA NAIP (17 U.S.C. 105) Cloud-Optimized GeoTIFF via STAC."""
    resp = requests.post(
        PUBLIC_NAIP_STAC_SEARCH_URL,
        json={
            "collections": ["naip"],
            "bbox": [min_lng, min_lat, max_lng, max_lat],
            "datetime": datetime_range,
            "limit": 10,
        },
        timeout=15,
    )
    resp.raise_for_status()
    feats = resp.json().get("features", [])
    if not feats:
        raise ValueError(f"No public-domain USDA NAIP COG found for bbox=({min_lng},{min_lat},{max_lng},{max_lat}).")
    feats.sort(key=lambda f: f["properties"].get("datetime", ""), reverse=True)
    chosen = feats[0]
    capture_dt = chosen["properties"]["datetime"][:10]
    raw_href = chosen["assets"]["image"]["href"]
    sign_resp = requests.get(PUBLIC_NAIP_STAC_SIGN_URL, params={"href": raw_href}, timeout=10)
    sign_resp.raise_for_status()
    return sign_resp.json()["href"], capture_dt


@contextlib.contextmanager
def _open_geotiff_reader(raster_uri: str):
    """Opens `gs://`, `https://`, or local GeoTIFFs via GDAL range reads (with `storage.Client` mTLS fallback)."""
    vsi_path = f"/vsigs/{raster_uri[5:]}" if raster_uri.startswith("gs://") else raster_uri
    try:
        with _rasterio_gcs_env(raster_uri), rasterio.open(vsi_path) as ds:
            yield ds
            return
    except Exception:
        if not raster_uri.startswith("gs://"):
            raise
    bkt_name, blob_path = raster_uri[5:].split("/", 1)
    raw_bytes = storage.Client(project=PROJECT_ID).bucket(bkt_name).blob(blob_path).download_as_bytes()
    with rasterio.io.MemoryFile(raw_bytes) as memfile, memfile.open() as ds:
        yield ds


def tile_geotiff_native_gsd(
    raster_uri: str,
    tile_px: int = 512,
    zone_label: str = "Custom GeoTIFF AOI",
    capture_date_iso: str = "2024-01-01",
    min_std: float = 6.0,
    max_tiles: Optional[int] = None,
    aoi_bbox_wgs84: Optional[Tuple[float, float, float, float]] = None,
) -> List[Dict[str, Any]]:
    """Slices a Cloud-Optimized GeoTIFF (`gs://...`, HTTPS COG, or local `.tif`) at 100% native sensor GSD."""
    tiles: List[Dict[str, Any]] = []
    with _open_geotiff_reader(raster_uri) as src:
        c_start, r_start = 0, 0
        c_end_lim, r_end_lim = src.width // tile_px, src.height // tile_px
        if aoi_bbox_wgs84 is not None:
            b_left, b_bottom, b_right, b_top = transform_bounds(
                "EPSG:4326", src.crs, *aoi_bbox_wgs84, densify_pts=4
            )
            r0_px, c0_px = src.index(b_left, b_top)
            r1_px, c1_px = src.index(b_right, b_bottom)
            r_min_px, r_max_px = max(0, min(r0_px, r1_px)), min(src.height, max(r0_px, r1_px))
            c_min_px, c_max_px = max(0, min(c0_px, c1_px)), min(src.width, max(c0_px, c1_px))
            r_start = r_min_px // tile_px
            c_start = c_min_px // tile_px
            r_end_lim = max(r_start + 1, math.ceil(r_max_px / tile_px))
            c_end_lim = max(c_start + 1, math.ceil(c_max_px / tile_px))
            r_end_lim = min(src.height // tile_px, r_end_lim)
            c_end_lim = min(src.width // tile_px, c_end_lim)

        band_indices = [1, 2, 3] if src.count >= 3 else [1, 1, 1]
        for r in range(r_start, r_end_lim):
            for c in range(c_start, c_end_lim):
                if max_tiles is not None and len(tiles) >= max_tiles:
                    return tiles
                win = Window(c * tile_px, r * tile_px, tile_px, tile_px)
                arr = src.read(band_indices, window=win)
                if arr.dtype != np.uint8:
                    arr = np.clip(arr, 0, 255).astype(np.uint8)
                rgb = np.transpose(arr, (1, 2, 0))
                pil_img = Image.fromarray(rgb, mode="RGB")
                if ImageStat.Stat(pil_img.convert("L")).stddev[0] < min_std:
                    continue
                left, bottom, right, top = rasterio.windows.bounds(win, src.transform)
                min_lng, min_lat, max_lng, max_lat = transform_bounds(
                    src.crs, "EPSG:4326", left, bottom, right, top, densify_pts=4
                )
                rel_r, rel_c = r - r_start, c - c_start
                tiles.append(
                    {
                        "tile_id": f"r{rel_r:03d}_c{rel_c:03d}",
                        "row_idx": rel_r,
                        "col_idx": rel_c,
                        "capture_date": capture_date_iso,
                        "zone": zone_label,
                        "min_lat": round(min_lat, 7),
                        "min_lng": round(min_lng, 7),
                        "max_lat": round(max_lat, 7),
                        "max_lng": round(max_lng, 7),
                        "center_lat": round((min_lat + max_lat) / 2.0, 7),
                        "center_lng": round((min_lng + max_lng) / 2.0, 7),
                        "wkt_geom": _bbox_wkt_polygon(min_lng, min_lat, max_lng, max_lat),
                        "pil_image": pil_img,
                        "gcs_uri": "",
                    }
                )
    return tiles


def tile_usgs_naip_aoi_grid(
    min_lng: float,
    min_lat: float,
    max_lng: float,
    max_lat: float,
    tile_px: int = 512,
    gsd_m: float = 0.6,
    zone_label: str = "USGS NAIP AOI",
    capture_date_iso: str = "2022-05-11",
    block_tiles: int = 4,
    min_std: float = 6.0,
    max_tiles: Optional[int] = None,
) -> List[Dict[str, Any]]:
    """Slices public-domain USDA NAIP at native 0.6m GSD via STAC Cloud-Optimized GeoTIFFs (with USGS ImageServer fallback)."""
    try:
        cog_href, cog_dt = discover_public_naip_cog_stac(min_lng, min_lat, max_lng, max_lat)
        stac_tiles = tile_geotiff_native_gsd(
            cog_href,
            tile_px=tile_px,
            zone_label=zone_label,
            capture_date_iso=cog_dt or capture_date_iso,
            min_std=min_std,
            max_tiles=max_tiles,
            aoi_bbox_wgs84=(min_lng, min_lat, max_lng, max_lat),
        )
        if stac_tiles:
            return stac_tiles
    except Exception as stac_exc:
        print(f"  ℹ️ STAC COG lookup note ({stac_exc}); falling back to USGSNAIPImagery ImageServer...")

    mid_lat = (min_lat + max_lat) / 2.0
    step_merc_m = (tile_px * gsd_m) / math.cos(math.radians(mid_lat))
    x0, x1 = _lng_to_merc_x(min_lng), _lng_to_merc_x(max_lng)
    y0, y1 = _lat_to_merc_y(min_lat), _lat_to_merc_y(max_lat)
    n_cols = max(1, math.ceil((x1 - x0) / step_merc_m))
    n_rows = max(1, math.ceil((y1 - y0) / step_merc_m))

    tiles: List[Dict[str, Any]] = []
    for br in range(0, n_rows, block_tiles):
        for bc in range(0, n_cols, block_tiles):
            if max_tiles is not None and len(tiles) >= max_tiles:
                return tiles
            r_end = min(n_rows, br + block_tiles)
            c_end = min(n_cols, bc + block_tiles)
            bw, bh = (c_end - bc), (r_end - br)
            xmin_b, xmax_b = x0 + bc * step_merc_m, x0 + c_end * step_merc_m
            ymax_b, ymin_b = y1 - br * step_merc_m, y1 - r_end * step_merc_m
            params = {
                "bbox": f"{xmin_b},{ymin_b},{xmax_b},{ymax_b}",
                "bboxSR": "3857",
                "imageSR": "3857",
                "size": f"{bw * tile_px},{bh * tile_px}",
                "format": "jpg",
                "compressionQuality": "92",
                "f": "image",
            }
            resp = requests.get(USGS_NAIP_IMAGERY_EXPORT_URL, params=params, timeout=60)
            resp.raise_for_status()
            super_img = Image.open(io.BytesIO(resp.content)).convert("RGB")
            for r in range(br, r_end):
                for c in range(bc, c_end):
                    if max_tiles is not None and len(tiles) >= max_tiles:
                        return tiles
                    dr, dc = r - br, c - bc
                    crop = super_img.crop((dc * tile_px, dr * tile_px, (dc + 1) * tile_px, (dr + 1) * tile_px))
                    if ImageStat.Stat(crop.convert("L")).stddev[0] < min_std:
                        continue
                    tx0, tx1 = x0 + c * step_merc_m, x0 + (c + 1) * step_merc_m
                    ty1, ty0 = y1 - r * step_merc_m, y1 - (r + 1) * step_merc_m
                    t_min_lng, t_max_lng = _merc_x_to_lng(tx0), _merc_x_to_lng(tx1)
                    t_min_lat, t_max_lat = _merc_y_to_lat(ty0), _merc_y_to_lat(ty1)
                    tiles.append(
                        {
                            "tile_id": f"r{r:03d}_c{c:03d}",
                            "row_idx": r,
                            "col_idx": c,
                            "capture_date": capture_date_iso,
                            "zone": zone_label,
                            "min_lat": round(t_min_lat, 7),
                            "min_lng": round(t_min_lng, 7),
                            "max_lat": round(t_max_lat, 7),
                            "max_lng": round(t_max_lng, 7),
                            "center_lat": round((t_min_lat + t_max_lat) / 2.0, 7),
                            "center_lng": round((t_min_lng + t_max_lng) / 2.0, 7),
                            "wkt_geom": _bbox_wkt_polygon(t_min_lng, t_min_lat, t_max_lng, t_max_lat),
                            "pil_image": crop,
                            "gcs_uri": "",
                        }
                    )
    return tiles


def stage_batch_prediction_inputs(
    tiles: List[Dict[str, Any]],
    gcs_bucket_name: str,
    prefix: str = "batch_inference",
    instances_format: str = "file-list",
    shard_size: int = 250,
) -> str:
    """Stages chips to Cloud Storage + writes `file-list` (`input_uris.txt`) or strict `{"image"}` `.jsonl`."""
    storage_client = storage.Client(project=PROJECT_ID)
    bucket = storage_client.bucket(gcs_bucket_name)
    chip_uris: List[str] = []
    for t in tiles:
        chip_blob_path = f"{prefix}/chips/{t['tile_id']}.png"
        buf = io.BytesIO()
        t["pil_image"].save(buf, format="PNG")
        bucket.blob(chip_blob_path).upload_from_string(buf.getvalue(), content_type="image/png")
        t["gcs_uri"] = f"gs://{gcs_bucket_name}/{chip_blob_path}"
        chip_uris.append(t["gcs_uri"])

    if instances_format == "file-list":
        list_blob = f"{prefix}/inputs/input_uris.txt"
        bucket.blob(list_blob).upload_from_string("\n".join(chip_uris) + "\n", content_type="text/plain")
        return f"gs://{gcs_bucket_name}/{list_blob}"

    # Strict `{"image": b64}` schema required by `SigLip512ImageHandler.EXPECTED_KEYS == {"image"}`
    for shard_idx, start in enumerate(range(0, len(tiles), shard_size)):
        lines = [
            json.dumps({"image": utils.b64_png(t["pil_image"])})
            for t in tiles[start : start + shard_size]
        ]
        shard_blob = f"{prefix}/inputs/image_{shard_idx:04d}.jsonl"
        bucket.blob(shard_blob).upload_from_string("\n".join(lines) + "\n", content_type="application/jsonl")
    return f"gs://{gcs_bucket_name}/{prefix}/inputs/image*.jsonl"


def ensure_bigquery_vector_tables(tiles_fqn: str, queries_fqn: str) -> None:
    """Creates dataset + partitioned/clustered tile & query vector tables if they do not exist."""
    dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{BQ_DATASET}")
    dataset_ref.location = "US"
    bq_client.create_dataset(dataset_ref, exists_ok=True)
    bq_client.query(
        f"""
        CREATE TABLE IF NOT EXISTS `{tiles_fqn}` (
          tile_id STRING NOT NULL,
          row_idx INT64,
          col_idx INT64,
          capture_date DATE,
          zone STRING,
          min_lat FLOAT64,
          min_lng FLOAT64,
          max_lat FLOAT64,
          max_lng FLOAT64,
          center_lat FLOAT64,
          center_lng FLOAT64,
          gcs_uri STRING,
          geom GEOGRAPHY,
          image_embedding ARRAY<FLOAT64>
        )
        PARTITION BY capture_date
        CLUSTER BY zone, tile_id
        """
    ).result()
    bq_client.query(
        f"""
        CREATE TABLE IF NOT EXISTS `{queries_fqn}` (
          query_id STRING NOT NULL,
          label STRING,
          prompt STRING,
          text_embedding ARRAY<FLOAT64>
        )
        """
    ).result()


def ingest_batch_predictions_to_bigquery(
    gcs_output_prefix: str, tiles: List[Dict[str, Any]], tiles_fqn: str
) -> int:
    """Loads `file-list` `BatchPredictionJob` output `.jsonl` shards into BigQuery `tiles_fqn`."""
    ensure_bigquery_vector_tables(tiles_fqn, QUERIES_TABLE_FQN)
    meta_by_uri = {t["gcs_uri"]: t for t in tiles if t.get("gcs_uri")}
    meta_by_id = {t["tile_id"]: t for t in tiles}
    bucket_name, blob_prefix = gcs_output_prefix[5:].split("/", 1)
    storage_client = storage.Client(project=PROJECT_ID)
    rows_to_load: List[Dict[str, Any]] = []
    for blob in storage_client.list_blobs(bucket_name, prefix=blob_prefix):
        if "prediction.results" not in blob.name and not blob.name.endswith(".jsonl"):
            continue
        for line in blob.download_as_text().splitlines():
            if not line.strip():
                continue
            rec = json.loads(line)
            inst = rec.get("instance")
            pred = rec.get("prediction", {})
            emb = pred.get("image_embedding")
            m = None
            if isinstance(inst, str):
                m = meta_by_uri.get(inst)
                if m is None:
                    stem = inst.rsplit("/", 1)[-1].rsplit(".", 1)[0]
                    m = meta_by_id.get(stem)
            if m is not None and emb:
                rows_to_load.append(
                    {
                        k: m[k]
                        for k in (
                            "tile_id",
                            "row_idx",
                            "col_idx",
                            "capture_date",
                            "zone",
                            "min_lat",
                            "min_lng",
                            "max_lat",
                            "max_lng",
                            "center_lat",
                            "center_lng",
                            "gcs_uri",
                        )
                    }
                    | {"geom": m["wkt_geom"], "image_embedding": [round(float(x), 6) for x in emb]}
                )
    if rows_to_load:
        bq_client.load_table_from_json(rows_to_load, tiles_fqn).result()
    return len(rows_to_load)


def embed_online_and_load_bigquery(
    tiles: List[Dict[str, Any]],
    queries_catalog: List[Dict[str, str]],
    active_endpoint: aiplatform.Endpoint,
    tiles_fqn: str,
    queries_fqn: str,
    chunk_size: int = 8,
    create_index: bool = False,
) -> None:
    """Chunks tiles (<=8 chips / <1.5 MB per call) to an Online Endpoint and loads BigQuery."""
    ensure_bigquery_vector_tables(tiles_fqn, queries_fqn)
    allowed = [1, 2, 4, 8]
    chunk_size = max([b for b in allowed if b <= max(1, chunk_size)])
    tile_rows: List[Dict[str, Any]] = []
    for start in range(0, len(tiles), chunk_size):
        chunk = tiles[start : start + chunk_size]
        req_bs = min([b for b in allowed if b >= len(chunk)] or [8])
        resp = active_endpoint.predict(
            instances=[{"image": utils.b64_png(t["pil_image"])} for t in chunk],
            parameters={"batch_size": req_bs},
            use_dedicated_endpoint=USE_DEDICATED_ENDPOINT,
        )
        for t, pred in zip(chunk, resp.predictions):
            emb = [round(float(x), 6) for x in pred["image_embedding"]]
            tile_rows.append(
                {
                    "tile_id": t["tile_id"],
                    "row_idx": t["row_idx"],
                    "col_idx": t["col_idx"],
                    "capture_date": t["capture_date"],
                    "zone": t["zone"],
                    "min_lat": t["min_lat"],
                    "min_lng": t["min_lng"],
                    "max_lat": t["max_lat"],
                    "max_lng": t["max_lng"],
                    "center_lat": t["center_lat"],
                    "center_lng": t["center_lng"],
                    "gcs_uri": t["gcs_uri"],
                    "geom": t["wkt_geom"],
                    "image_embedding": emb,
                }
            )
    bq_client.load_table_from_json(tile_rows, tiles_fqn).result()

    allowed_txt = [1, 2, 4, 8, 16, 32]
    txt_preds: List[Dict[str, Any]] = []
    for start in range(0, len(queries_catalog), 16):
        q_chunk = queries_catalog[start : start + 16]
        q_bs = min([b for b in allowed_txt if b >= len(q_chunk)] or [32])
        txt_resp = active_endpoint.predict(
            instances=[{"text": q["prompt"]} for q in q_chunk],
            parameters={"batch_size": q_bs},
            use_dedicated_endpoint=USE_DEDICATED_ENDPOINT,
        )
        txt_preds.extend(txt_resp.predictions)

    query_rows = [
        {
            "query_id": q["query_id"],
            "label": q["label"],
            "prompt": q["prompt"],
            "text_embedding": [round(float(x), 6) for x in pred["text_embedding"]],
        }
        for q, pred in zip(queries_catalog, txt_preds)
    ]
    bq_client.load_table_from_json(
        query_rows,
        queries_fqn,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
    ).result()

    if create_index:
        bq_client.query(
            f"""
            CREATE VECTOR INDEX IF NOT EXISTS `{TILES_TABLE_NAME}_ivf_idx`
            ON `{tiles_fqn}`(image_embedding)
            OPTIONS(index_type = 'IVF', distance_type = 'COSINE')
            """
        ).result()


sliced_tiles: List[Dict[str, Any]] = []
batch_input_uri: str = ""
if BUILD_TILES_FROM_SCRATCH:
    if SOURCE_GEOTIFF_URI:
        print(f"🛰️ Slicing GeoTIFF `{SOURCE_GEOTIFF_URI}` at native sensor GSD ({TILE_SIZE_PX}x{TILE_SIZE_PX} px)...")
        sliced_tiles = tile_geotiff_native_gsd(
            SOURCE_GEOTIFF_URI,
            tile_px=int(TILE_SIZE_PX),
            zone_label=AOI_NAME,
            capture_date_iso=CAPTURE_DATE_ISO,
            min_std=float(MIN_TILE_STDDEV),
            max_tiles=int(MAX_DEMO_TILES),
        )
    else:
        print(f"🛰️ Slicing public-domain USGS NAIP grid for `{AOI_NAME}` ({TILE_SIZE_PX}x{TILE_SIZE_PX} px @ {TARGET_GSD_M}m GSD)...")
        sliced_tiles = tile_usgs_naip_aoi_grid(
            AOI_MIN_LNG,
            AOI_MIN_LAT,
            AOI_MAX_LNG,
            AOI_MAX_LAT,
            tile_px=int(TILE_SIZE_PX),
            gsd_m=float(TARGET_GSD_M),
            zone_label=AOI_NAME,
            capture_date_iso=CAPTURE_DATE_ISO,
            min_std=float(MIN_TILE_STDDEV),
            max_tiles=int(MAX_DEMO_TILES),
        )
    print(f"✅ Sliced {len(sliced_tiles)} non-uniform tiles (filtered grayscale stddev >= {MIN_TILE_STDDEV}).")

    if GCS_BUCKET_NAME and sliced_tiles:
        batch_input_uri = stage_batch_prediction_inputs(
            sliced_tiles, GCS_BUCKET_NAME, instances_format=INPUT_SOURCE_FORMAT
        )
        print(f"📦 Staged Cloud Storage chips & Batch Prediction inputs ({INPUT_SOURCE_FORMAT}): {batch_input_uri}")

    if endpoint is not None and sliced_tiles:
        print(f"⚡ Embedding {len(sliced_tiles)} tiles + {len(DEFAULT_DOMAIN_QUERIES)} queries into BigQuery...")
        embed_online_and_load_bigquery(
            sliced_tiles,
            DEFAULT_DOMAIN_QUERIES,
            endpoint,
            TILES_TABLE_FQN,
            QUERIES_TABLE_FQN,
            chunk_size=int(BATCH_SIZE),
            create_index=bool(CREATE_VECTOR_INDEX),
        )
        print(f"✅ Loaded `{TILES_TABLE_FQN}` and `{QUERIES_TABLE_FQN}`!")

if RUN_BATCH_JOB_EXAMPLE and GCS_BUCKET_NAME:
    batch_model = utils.create_model(
        display_name=f"{DISPLAY_NAME}-{MODEL_TYPE.lower()}-batch-{unique_suffix}",
        model_type=MODEL_TYPE,
        model_mode="IMAGE_ONLY" if INPUT_SOURCE_FORMAT == "file-list" else MODEL_MODE,
        accelerator=ACCELERATOR,
        batch_model=True,
    )
    platform_config = utils.PLATFORM_CONFIGS[ACCELERATOR]
    resolved_input_uri = batch_input_uri or (
        f"gs://{GCS_BUCKET_NAME}/batch_inference/inputs/input_uris.txt"
        if INPUT_SOURCE_FORMAT == "file-list"
        else f"gs://{GCS_BUCKET_NAME}/batch_inference/inputs/image*.jsonl"
    )
    batch_output_uri = f"gs://{GCS_BUCKET_NAME}/batch_inference/outputs/{unique_suffix}/"
    job_display_name = f"{DISPLAY_NAME}-batch-{unique_suffix}"
    print(f"📦 Launching BatchPredictionJob ({INPUT_SOURCE_FORMAT}) -> {batch_output_uri}")
    batch_job = batch_model.batch_predict(
        job_display_name=job_display_name,
        gcs_source=resolved_input_uri,
        gcs_destination_prefix=batch_output_uri,
        instances_format=INPUT_SOURCE_FORMAT,
        machine_type=platform_config.machine_type,
        accelerator_count=platform_config.accelerator_count,
        accelerator_type=platform_config.accelerator_type,
        starting_replica_count=MIN_REPLICA_COUNT,
        max_replica_count=MAX_REPLICA_COUNT,
        labels={
            "task": "batch-inference",
            "vertex-ai-pipelines-run-billing-id": job_display_name,
        },
        batch_size=BATCH_SIZE,
        sync=False,
    )
    print(f"✅ Started BatchPredictionJob: {batch_job.resource_name}")
elif not BUILD_TILES_FROM_SCRATCH:
    print(
        "ℹ️ `BUILD_TILES_FROM_SCRATCH = False` — native-GSD GeoTIFF & USGS NAIP tiling helpers "
        "(`tile_geotiff_native_gsd`, `tile_usgs_naip_aoi_grid`, `stage_batch_prediction_inputs`, "
        "`ingest_batch_predictions_to_bigquery`, `embed_online_and_load_bigquery`) are defined and ready."
    )

## Step 4: Inspect the BigQuery Spatial & `1152-D` Embedding Tables

Once your regional tiles (`IMAGE_ONLY`) and domain search prompts (`TEXT_ONLY`) are encoded by **`ASM - Classification and Retrieval` (`RS-SigLIP2`)**, they live in two BigQuery tables:

1. **`region_tiles_*` (`TILES_TABLE_FQN`)**:
   - `tile_id`, `row_idx`, `col_idx`: Grid coordinates across the metropolitan AOI.
   - `center_lat`, `center_lng`, `min_lat`, `min_lng`, `max_lat`, `max_lng`, `geom`: Exact WGS84 bounding box and `GEOGRAPHY` polygon of each `512×512 px` (`~307 m × 307 m` @ `0.6 m` GSD) tile.
   - `zone`: Sub-regional neighborhood label (e.g., *Port of LA / San Pedro*, *Port of Long Beach*, *LAX / El Segundo*).
   - `image_embedding`: `ARRAY<FLOAT64>` of length **`1,152`** ($\|v_{\text{img}}\|_2 = 1$).
2. **`region_queries_*` (`QUERIES_TABLE_FQN`)**:
   - `query_id`, `label`, `prompt`: Curated infrastructure, logistics, energy, and environmental text queries (including negative background nuisance anchors).
   - `text_embedding`: `ARRAY<FLOAT64>` of length **`1,152`** ($\|v_{\text{txt}}\|_2 = 1$).

If `FETCH_LIVE_NAIP_TILES = True` in Step 2, this step also previews a sample `512×512` public-domain orthoimagery tile directly from the federal **USGS National Map NAIP ImageServer (`USGSNAIPImagery`)**.

In [ ]:
# @title 4. Verify BigQuery Tile Coverage, Indexed Text Prompts & Optional Live USGS NAIP Tile Fetch
USGS_NAIP_EXPORT_URL = (
    "https://imagery.nationalmap.gov/arcgis/rest/services/USGSNAIPImagery/ImageServer/exportImage"
)


def fetch_usgs_naip_tile(
    min_lng: float, min_lat: float, max_lng: float, max_lat: float, size_px: int = 512
) -> Image.Image:
    """Fetches a 100% public-domain federal USDA/USGS NAIP orthoimagery tile (17 U.S.C. 105)."""
    try:
        cog_href, _ = discover_public_naip_cog_stac(min_lng, min_lat, max_lng, max_lat)
        tiles = tile_geotiff_native_gsd(
            cog_href, tile_px=size_px, min_std=0.0, max_tiles=1, aoi_bbox_wgs84=(min_lng, min_lat, max_lng, max_lat)
        )
        if tiles:
            return tiles[0]["pil_image"]
    except Exception:
        pass
    params = {
        "bbox": f"{_lng_to_merc_x(min_lng)},{_lat_to_merc_y(min_lat)},{_lng_to_merc_x(max_lng)},{_lat_to_merc_y(max_lat)}",
        "bboxSR": "3857",
        "imageSR": "3857",
        "size": f"{size_px},{size_px}",
        "format": "jpg",
        "compressionQuality": "92",
        "f": "image",
    }
    resp = requests.get(USGS_NAIP_EXPORT_URL, params=params, timeout=20)
    resp.raise_for_status()
    return Image.open(io.BytesIO(resp.content)).convert("RGB")


if FETCH_LIVE_NAIP_TILES:
    print(f"🛰️ Fetching sample {TILE_SIZE_PX}x{TILE_SIZE_PX} public-domain USGS NAIP tile for {AOI_NAME}...")
    sample_step_m = (TILE_SIZE_PX * TARGET_GSD_M) / math.cos(math.radians(AOI_MIN_LAT))
    sx0 = _lng_to_merc_x(AOI_MIN_LNG)
    sy0 = _lat_to_merc_y(AOI_MIN_LAT)
    sample_naip_img = fetch_usgs_naip_tile(
        AOI_MIN_LNG,
        AOI_MIN_LAT,
        _merc_x_to_lng(sx0 + sample_step_m),
        _merc_y_to_lat(sy0 + sample_step_m),
        int(TILE_SIZE_PX),
    )
    plt.figure(figsize=(4.5, 4.5))
    plt.imshow(sample_naip_img)
    plt.title(f"Sample USGS NAIP Tile ({AOI_NAME})", fontsize=11, fontweight="bold")
    plt.axis("off")
    plt.show()

zone_summary_sql = f"""
SELECT
  zone,
  COUNT(*) AS tile_count,
  ROUND(COUNT(*) * 0.3072 * 0.3072, 1) AS area_km2,
  ROUND(AVG(ARRAY_LENGTH(image_embedding)), 0) AS embedding_dims,
  ROUND(MIN(min_lat), 4) AS south_lat,
  ROUND(MAX(max_lat), 4) AS north_lat,
  ROUND(MIN(min_lng), 4) AS west_lng,
  ROUND(MAX(max_lng), 4) AS east_lng
FROM `{TILES_TABLE_FQN}`
GROUP BY zone
ORDER BY tile_count DESC
"""

zone_df = bq_client.query(zone_summary_sql).to_dataframe()
total_tiles = int(zone_df["tile_count"].sum())
total_km2 = float(zone_df["area_km2"].sum())
print(
    f"📦 `{TILES_TABLE_FQN}`: {total_tiles:,} indexed tiles (~{total_km2:,.1f} km²) across {len(zone_df)} zones:"
)
display(zone_df)

queries_sql = f"""
SELECT
  query_id,
  label,
  prompt,
  ARRAY_LENGTH(text_embedding) AS embedding_dims
FROM `{QUERIES_TABLE_FQN}`
ORDER BY query_id
"""
queries_df = bq_client.query(queries_sql).to_dataframe()
print(f"\n🔎 `{QUERIES_TABLE_FQN}`: {len(queries_df)} pre-encoded 1152-D text embeddings:")
display(queries_df)

## Step 5: Three Metro-Scale BigQuery Retrieval Patterns

Because both images and text share the same L2-normalized `1152-D` `RS-SigLIP2` vector space in BigQuery, we can execute three distinct geospatial intelligence patterns in pure SQL without spinning up a GPU at query time:

| Retrieval Mode | Input | BigQuery SQL Formula | Real-World Use Case |
| :--- | :--- | :--- | :--- |
| **5A. Text-to-Tile Semantic Search** | Text prompt vector $v_{\text{txt}}$ | `1.0 - ML.DISTANCE(t.image_embedding, q.text_embedding, 'COSINE')` | Find all distribution warehouses, gantry cranes, solar rooftops, or water treatment plants across a `1,650+ km²` metro region. |
| **5B. Tile-to-Tile Visual Similarity** | Seed tile vector $v_{\text{seed}}$ | `1.0 - ML.DISTANCE(t.image_embedding, s.image_embedding, 'COSINE')` | Click one known facility (e.g., an intermodal rail yard or terminal berth) and find every visually similar footprint across the metro grid. |
| **5C. Opportunity Finder (Vector Algebra)** | Positive prompt $v_{+}$ & Negative prompt $v_{-}$ | `score(v_+) - α * score(v_-)` (with regional z-score normalization) | **Solar Rooftop Prospecting**: Find large commercial warehouse roofs (`+distribution_warehouses`) that **do not yet** have solar arrays (`-0.5 * solar_rooftops`). |

Below, we execute all three retrieval patterns using the parameters configured in **Step 2**.

In [ ]:
# @title 5A. Text-to-Tile Semantic Search (`ML.DISTANCE` + Regional Z-Score) & Visual Tile Grid
text_search_sql = f"""
WITH raw_scores AS (
  SELECT
    t.tile_id,
    t.zone,
    t.row_idx,
    t.col_idx,
    ROUND(t.center_lat, 5) AS center_lat,
    ROUND(t.center_lng, 5) AS center_lng,
    t.gcs_uri,
    q.label AS query_label,
    q.prompt AS query_prompt,
    1.0 - ML.DISTANCE(t.image_embedding, q.text_embedding, 'COSINE') AS cosine_sim
  FROM `{TILES_TABLE_FQN}` AS t
  CROSS JOIN `{QUERIES_TABLE_FQN}` AS q
  WHERE q.query_id = @query_id
),
stats AS (
  SELECT
    AVG(cosine_sim) AS mean_sim,
    STDDEV(cosine_sim) AS std_sim
  FROM raw_scores
)
SELECT
  ROW_NUMBER() OVER (ORDER BY r.cosine_sim DESC) AS rank,
  r.tile_id,
  r.zone,
  r.center_lat,
  r.center_lng,
  ROUND(r.cosine_sim, 4) AS cosine_sim,
  ROUND(SAFE_DIVIDE(r.cosine_sim - s.mean_sim, s.std_sim), 2) AS z_score_sigma,
  r.gcs_uri,
  r.query_label,
  r.query_prompt
FROM raw_scores AS r
CROSS JOIN stats AS s
ORDER BY r.cosine_sim DESC
LIMIT @top_k
"""

job_config_5a = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("query_id", "STRING", SEARCH_QUERY_ID),
        bigquery.ScalarQueryParameter("top_k", "INT64", int(TOP_K_RESULTS)),
    ]
)
search_results_df = bq_client.query(text_search_sql, job_config=job_config_5a).to_dataframe()

query_title = search_results_df["query_label"].iloc[0] if not search_results_df.empty else SEARCH_QUERY_ID
query_prompt_str = search_results_df["query_prompt"].iloc[0] if not search_results_df.empty else ""
print(f"🔎 Mode 5A — Text-to-Tile Search for `{SEARCH_QUERY_ID}` ({query_title})")
print(f"   Prompt: '{query_prompt_str}'")
display(
    search_results_df[
        ["rank", "tile_id", "zone", "center_lat", "center_lng", "cosine_sim", "z_score_sigma"]
    ]
)


def plot_naip_tiles_from_gcs(df: pd.DataFrame, title: str, max_tiles: int = 6) -> None:
    """Renders USDA NAIP public-domain tiles from GCS (or live USGS fallback) if accessible."""
    subset = df.head(max_tiles)
    n = len(subset)
    if n == 0:
        return
    cols = min(3, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 4.5 * rows))
    axes_flat = np.array(axes).reshape(-1) if n > 1 else [axes]
    storage_client = storage.Client(project=PROJECT_ID)

    for idx, (_, row) in enumerate(subset.iterrows()):
        ax = axes_flat[idx]
        gcs_uri = str(row.get("gcs_uri", ""))
        img = None
        try:
            if gcs_uri.startswith("gs://"):
                parts = gcs_uri[5:].split("/", 1)
                bucket = storage_client.bucket(parts[0])
                blob = bucket.blob(parts[1])
                img_bytes = blob.download_as_bytes()
                img = Image.open(io.BytesIO(img_bytes))
        except Exception:
            img = None
        if img is None:
            try:
                c_lat, c_lng = float(row["center_lat"]), float(row["center_lng"])
                img = fetch_usgs_naip_tile(
                    c_lng - 0.00168, c_lat - 0.00138, c_lng + 0.00168, c_lat + 0.00138, 256
                )
            except Exception:
                img = None
        if img is not None:
            ax.imshow(img)
        else:
            ax.text(
                0.5,
                0.5,
                f"Tile {row['tile_id']}\n({row['zone']})\n({row['center_lat']}, {row['center_lng']})",
                ha="center",
                va="center",
                fontsize=10,
            )
            ax.set_facecolor("#F1F3F4")
        score_label = (
            f"cos={row['cosine_sim']:.3f} (+{row['z_score_sigma']:.1f}σ)"
            if "cosine_sim" in row
            else f"algebra={row['net_opportunity_sigma']:+.2f}σ"
        )
        ax.set_title(f"#{int(row['rank'])} {row['tile_id']} | {score_label}", fontsize=10, fontweight="bold")
        ax.set_xlabel(f"{row['zone']}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

    for extra_ax in axes_flat[n:]:
        extra_ax.axis("off")
    plt.suptitle(title, fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()


plot_naip_tiles_from_gcs(
    search_results_df,
    title=f"Top {min(int(VISUAL_GRID_TILES), len(search_results_df))} USDA NAIP Matches — {query_title}",
    max_tiles=int(VISUAL_GRID_TILES),
)

In [ ]:
# @title 5B. Tile-to-Tile Visual Similarity (`SEED_TILE_ID` Nearest Neighbors)
# If SEED_TILE_ID is not present in the active table, automatically fall back to Rank #1 from Step 5A
effective_seed_tile = (
    SEED_TILE_ID
    if not search_results_df.empty
    else "r038_c052"
)

tile_similarity_sql = f"""
WITH seed AS (
  SELECT
    tile_id AS seed_id,
    zone AS seed_zone,
    image_embedding AS seed_embedding
  FROM `{TILES_TABLE_FQN}`
  WHERE tile_id = @seed_tile_id
  LIMIT 1
)
SELECT
  ROW_NUMBER() OVER (
    ORDER BY 1.0 - ML.DISTANCE(t.image_embedding, s.seed_embedding, 'COSINE') DESC
  ) AS rank,
  t.tile_id,
  t.zone,
  ROUND(t.center_lat, 5) AS center_lat,
  ROUND(t.center_lng, 5) AS center_lng,
  ROUND(1.0 - ML.DISTANCE(t.image_embedding, s.seed_embedding, 'COSINE'), 4) AS cosine_sim,
  t.gcs_uri
FROM `{TILES_TABLE_FQN}` AS t
CROSS JOIN seed AS s
WHERE t.tile_id != s.seed_id
ORDER BY cosine_sim DESC
LIMIT @top_k
"""

job_config_5b = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("seed_tile_id", "STRING", effective_seed_tile),
        bigquery.ScalarQueryParameter("top_k", "INT64", int(TOP_K_RESULTS)),
    ]
)
similar_tiles_df = bq_client.query(tile_similarity_sql, job_config=job_config_5b).to_dataframe()

if similar_tiles_df.empty and not search_results_df.empty:
    effective_seed_tile = str(search_results_df["tile_id"].iloc[0])
    job_config_5b = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("seed_tile_id", "STRING", effective_seed_tile),
            bigquery.ScalarQueryParameter("top_k", "INT64", int(TOP_K_RESULTS)),
        ]
    )
    similar_tiles_df = bq_client.query(tile_similarity_sql, job_config=job_config_5b).to_dataframe()

print(f"🧭 Mode 5B — Top {len(similar_tiles_df)} Visually Similar Tiles to Seed Tile `{effective_seed_tile}`:")
display(similar_tiles_df[["rank", "tile_id", "zone", "center_lat", "center_lng", "cosine_sim"]])

In [ ]:
# @title 5C. Opportunity Finder — Vector Algebra (`POSITIVE_QUERY_ID - NEGATIVE_WEIGHT * NEGATIVE_QUERY_ID`)
vector_algebra_sql = f"""
WITH pos_q AS (
  SELECT label, text_embedding FROM `{QUERIES_TABLE_FQN}` WHERE query_id = @pos_id
),
neg_q AS (
  SELECT label, text_embedding FROM `{QUERIES_TABLE_FQN}` WHERE query_id = @neg_id
),
tile_scores AS (
  SELECT
    t.tile_id,
    t.zone,
    ROUND(t.center_lat, 5) AS center_lat,
    ROUND(t.center_lng, 5) AS center_lng,
    t.gcs_uri,
    p.label AS pos_label,
    n.label AS neg_label,
    1.0 - ML.DISTANCE(t.image_embedding, p.text_embedding, 'COSINE') AS pos_cos,
    1.0 - ML.DISTANCE(t.image_embedding, n.text_embedding, 'COSINE') AS neg_cos
  FROM `{TILES_TABLE_FQN}` AS t
  CROSS JOIN pos_q AS p
  CROSS JOIN neg_q AS n
),
stats AS (
  SELECT
    AVG(pos_cos) AS mean_pos, STDDEV(pos_cos) AS std_pos,
    AVG(neg_cos) AS mean_neg, STDDEV(neg_cos) AS std_neg
  FROM tile_scores
),
normalized AS (
  SELECT
    ts.*,
    SAFE_DIVIDE(ts.pos_cos - s.mean_pos, s.std_pos) AS pos_z,
    SAFE_DIVIDE(ts.neg_cos - s.mean_neg, s.std_neg) AS neg_z,
    SAFE_DIVIDE(ts.pos_cos - s.mean_pos, s.std_pos)
      - (@neg_weight * SAFE_DIVIDE(ts.neg_cos - s.mean_neg, s.std_neg)) AS net_opportunity_sigma
  FROM tile_scores AS ts
  CROSS JOIN stats AS s
)
SELECT
  ROW_NUMBER() OVER (ORDER BY net_opportunity_sigma DESC) AS rank,
  tile_id,
  zone,
  center_lat,
  center_lng,
  ROUND(pos_cos, 4) AS pos_cosine,
  ROUND(pos_z, 2) AS pos_sigma,
  ROUND(neg_cos, 4) AS neg_cosine,
  ROUND(neg_z, 2) AS neg_sigma,
  ROUND(net_opportunity_sigma, 2) AS net_opportunity_sigma,
  gcs_uri
FROM normalized
ORDER BY net_opportunity_sigma DESC
LIMIT @top_k
"""

job_config_5c = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("pos_id", "STRING", POSITIVE_QUERY_ID),
        bigquery.ScalarQueryParameter("neg_id", "STRING", NEGATIVE_QUERY_ID),
        bigquery.ScalarQueryParameter("neg_weight", "FLOAT64", float(NEGATIVE_WEIGHT)),
        bigquery.ScalarQueryParameter("top_k", "INT64", int(TOP_K_RESULTS)),
    ]
)
algebra_df = bq_client.query(vector_algebra_sql, job_config=job_config_5c).to_dataframe()

print(
    f"⚡ Mode 5C — Opportunity Finder Vector Algebra: "
    f"`+{POSITIVE_QUERY_ID}` minus `{NEGATIVE_WEIGHT:.1f} × {NEGATIVE_QUERY_ID}`"
)
display(
    algebra_df[
        [
            "rank",
            "tile_id",
            "zone",
            "center_lat",
            "center_lng",
            "pos_sigma",
            "neg_sigma",
            "net_opportunity_sigma",
        ]
    ]
)

## Step 6: Interactive Google Maps Visualization (`google.maps.Map`)

Next, let's render the top overhead matches directly on an interactive **Google Map** (`maps.googleapis.com/maps/api/js` with `mapTypeId: "hybrid"`) so you can inspect each `307 m × 307 m` tile footprint:
- **Red (`#D93025`) Bounding Boxes**: Top **Text-to-Tile (`SEARCH_QUERY_ID`)** semantic search matches from Step 5A.
- **Green (`#1E8E3E`) Bounding Boxes**: Top **Opportunity Finder (`POSITIVE_QUERY_ID - NEGATIVE_WEIGHT * NEGATIVE_QUERY_ID`)** vector algebra matches from Step 5C.
- Click any tile rectangle on the Google Map to open its `google.maps.InfoWindow` details card, or use the top-right layer checkboxes to toggle layers on and off.

In [ ]:
# @title 6. Render Interactive Google Map (`google.maps.Map`) of Top Matches
if not GMP_API_KEY:
    print(
        "⚠️ Please set `GMP_API_KEY` in Step 2 (or in Colab Secrets) to render the interactive Google Map."
    )
elif not search_results_df.empty:
    half_deg_lat = 0.00138  # ~153.6m half-tile height
    half_deg_lng = 0.00168  # ~153.6m half-tile width at ~34 deg latitude

    search_features = []
    for _, row in search_results_df.iterrows():
        c_lat, c_lng = float(row["center_lat"]), float(row["center_lng"])
        search_features.append(
            {
                "rank": int(row["rank"]),
                "tile_id": str(row["tile_id"]),
                "zone": str(row["zone"]),
                "north": c_lat + half_deg_lat,
                "south": c_lat - half_deg_lat,
                "east": c_lng + half_deg_lng,
                "west": c_lng - half_deg_lng,
                "center_lat": c_lat,
                "center_lng": c_lng,
                "cosine_sim": float(row["cosine_sim"]),
                "z_score_sigma": float(row["z_score_sigma"]),
            }
        )

    algebra_features = []
    if not algebra_df.empty:
        for _, row in algebra_df.iterrows():
            c_lat, c_lng = float(row["center_lat"]), float(row["center_lng"])
            algebra_features.append(
                {
                    "rank": int(row["rank"]),
                    "tile_id": str(row["tile_id"]),
                    "zone": str(row["zone"]),
                    "north": c_lat + half_deg_lat,
                    "south": c_lat - half_deg_lat,
                    "east": c_lng + half_deg_lng,
                    "west": c_lng - half_deg_lng,
                    "center_lat": c_lat,
                    "center_lng": c_lng,
                    "pos_sigma": float(row["pos_sigma"]),
                    "neg_sigma": float(row["neg_sigma"]),
                    "net_opportunity_sigma": float(row["net_opportunity_sigma"]),
                }
            )

    map_div_id = f"gmap_{uuid.uuid4().hex[:8]}"
    gmap_html = f"""
    <div style="position: relative; width: 100%; height: 600px; border-radius: 12px; overflow: hidden; border: 1px solid #DADCE0; font-family: 'Google Sans', Roboto, Arial, sans-serif;">
      <div id="{map_div_id}" style="width: 100%; height: 100%;"></div>
      <div style="position: absolute; top: 12px; right: 60px; background: rgba(255, 255, 255, 0.95); padding: 10px 14px; border-radius: 8px; box-shadow: 0 2px 6px rgba(0,0,0,0.25); font-size: 13px; color: #202124; z-index: 5;">
        <div style="font-weight: 600; margin-bottom: 6px;">Map Layers</div>
        <label style="display: flex; align-items: center; gap: 6px; cursor: pointer; margin-bottom: 4px;">
          <input type="checkbox" id="{map_div_id}_chk_search" checked>
          <span style="display:inline-block;width:12px;height:12px;background:#D93025;border:1px solid #B31412;border-radius:2px;"></span>
          5A: Text Search (<code>{SEARCH_QUERY_ID}</code>)
        </label>
        <label style="display: flex; align-items: center; gap: 6px; cursor: pointer;">
          <input type="checkbox" id="{map_div_id}_chk_algebra" checked>
          <span style="display:inline-block;width:12px;height:12px;background:#1E8E3E;border:1px solid #137333;border-radius:2px;"></span>
          5C: Opportunity Finder (<code>+{POSITIVE_QUERY_ID} - {NEGATIVE_WEIGHT}*{NEGATIVE_QUERY_ID}</code>)
        </label>
      </div>
    </div>
    <script>
      (function() {{
        const searchItems = {json.dumps(search_features)};
        const algebraItems = {json.dumps(algebra_features)};

        function initMap() {{
          const map = new google.maps.Map(document.getElementById("{map_div_id}"), {{
            center: {{ lat: searchItems[0].center_lat, lng: searchItems[0].center_lng }},
            zoom: 12,
            mapTypeId: "hybrid",
            mapTypeControl: true,
            streetViewControl: false,
            fullscreenControl: true
          }});

          const infoWindow = new google.maps.InfoWindow();
          const bounds = new google.maps.LatLngBounds();
          const searchRects = [];
          const algebraRects = [];

          searchItems.forEach((item) => {{
            const rectBounds = {{
              north: item.north,
              south: item.south,
              east: item.east,
              west: item.west
            }};
            bounds.extend({{ lat: item.north, lng: item.east }});
            bounds.extend({{ lat: item.south, lng: item.west }});

            const rect = new google.maps.Rectangle({{
              bounds: rectBounds,
              strokeColor: "#D93025",
              strokeOpacity: 0.95,
              strokeWeight: 2,
              fillColor: "#D93025",
              fillOpacity: 0.28,
              map: map
            }});

            rect.addListener("click", () => {{
              infoWindow.setContent(
                `<div style="font-family: Roboto, Arial, sans-serif; font-size: 13px; line-height: 1.45;">` +
                `<b>#${{item.rank}} Tile ${{item.tile_id}}</b><br>` +
                `Zone: ${{item.zone}}<br>` +
                `Query: <code>{SEARCH_QUERY_ID}</code><br>` +
                `Cosine Sim: <b>${{item.cosine_sim.toFixed(4)}}</b> (+${{item.z_score_sigma.toFixed(2)}}&sigma;)` +
                `</div>`
              );
              infoWindow.setPosition({{ lat: item.center_lat, lng: item.center_lng }});
              infoWindow.open(map);
            }});
            searchRects.push(rect);
          }});

          algebraItems.forEach((item) => {{
            const rectBounds = {{
              north: item.north,
              south: item.south,
              east: item.east,
              west: item.west
            }};
            bounds.extend({{ lat: item.north, lng: item.east }});
            bounds.extend({{ lat: item.south, lng: item.west }});

            const rect = new google.maps.Rectangle({{
              bounds: rectBounds,
              strokeColor: "#1E8E3E",
              strokeOpacity: 0.95,
              strokeWeight: 2,
              fillColor: "#1E8E3E",
              fillOpacity: 0.24,
              map: map
            }});

            rect.addListener("click", () => {{
              infoWindow.setContent(
                `<div style="font-family: Roboto, Arial, sans-serif; font-size: 13px; line-height: 1.45;">` +
                `<b>Opportunity #${{item.rank}}: ${{item.tile_id}}</b><br>` +
                `Zone: ${{item.zone}}<br>` +
                `Net Opportunity: <b>+${{item.net_opportunity_sigma.toFixed(2)}}&sigma;</b><br>` +
                `(+{POSITIVE_QUERY_ID}: ${{item.pos_sigma.toFixed(2)}}&sigma; | -{NEGATIVE_QUERY_ID}: ${{item.neg_sigma.toFixed(2)}}&sigma;)` +
                `</div>`
              );
              infoWindow.setPosition({{ lat: item.center_lat, lng: item.center_lng }});
              infoWindow.open(map);
            }});
            algebraRects.push(rect);
          }});

          if (!bounds.isEmpty()) {{
            map.fitBounds(bounds);
          }}

          document.getElementById("{map_div_id}_chk_search").addEventListener("change", (e) => {{
            searchRects.forEach((r) => r.setMap(e.target.checked ? map : null));
          }});
          document.getElementById("{map_div_id}_chk_algebra").addEventListener("change", (e) => {{
            algebraRects.forEach((r) => r.setMap(e.target.checked ? map : null));
          }});
        }}

        if (window.google && window.google.maps) {{
          initMap();
        }} else {{
          const script = document.createElement("script");
          script.src = "https://maps.googleapis.com/maps/api/js?key={GMP_API_KEY}";
          script.async = true;
          script.onload = initMap;
          document.head.appendChild(script);
        }}
      }})();
    </script>
    """
    display(HTML(gmap_html))

## Step 7: Optional Resource Cleanup

If you deployed a dedicated Online Prediction Endpoint in **Step 3** (`DEPLOY_LIVE_ENDPOINT = True`), undeploying or deleting the endpoint when your session finishes avoids idle GPU billing (`NVIDIA_L4` / `A100`). Control this behavior using `UNDEPLOY_ENDPOINT_ON_FINISH` and `DELETE_ENDPOINT_AND_MODEL` in **Step 2**.

In [ ]:
# @title 7. Undeploy or Delete Endpoint (Controlled via Step 2 Parameters)
if endpoint is not None and UNDEPLOY_ENDPOINT_ON_FINISH:
    if DELETE_ENDPOINT_AND_MODEL:
        print(f"🧹 Deleting endpoint {endpoint.resource_name} (force=True) and model...")
        endpoint.delete(force=True)
        if model is not None:
            model.delete()
        print("✅ Deleted Endpoint and Model resource.")
    else:
        print(f"🧹 Undeploying all models from {endpoint.resource_name}...")
        endpoint.undeploy_all()
        print("✅ Undeployed GPU replicas (Endpoint metadata retained).")
else:
    print("ℹ️ No Endpoint teardown requested (or running in Pre-computed BigQuery Mode).")